In [1]:
from IPython.display import HTML, display

html_content = """
<div style='font-family:Segoe UI; padding:20px; background:#f8f9fa; border-radius:10px;'>

<h2 style='color:#0d6efd;'>📚 AI Book Reader and Idea Generator</h2>

<p>
This program acts like a smart digital assistant that can read a book,
understand its content, and answer questions about it.
</p>

<h3>🔹 Step 1: Load Secret Keys</h3>
<p>
The program reads secret API keys from a file. These keys allow it to connect
to AI services and AgentOps monitoring.
</p>

<h3>🔹 Step 2: Create an AI Assistant</h3>
<p>
An AI Assistant is created using GPT/Gemini models. This assistant can
understand questions and generate intelligent responses.
</p>

<h3>🔹 Step 3: Read a PDF Book</h3>
<p>
The application opens a PDF book and extracts its text content so the AI can
understand what's written inside.
</p>

<h3>🔹 Step 4: Store Knowledge in ChromaDB</h3>
<p>
The book content is stored inside a vector database called ChromaDB.
Think of it as a smart searchable memory for the AI.
</p>

<h3>🔹 Step 5: Create a Retrieval Agent (RAG)</h3>
<p>
A special agent called RAG (Retrieval Augmented Generation) is created.
Instead of guessing answers, it first searches the book and then answers using
the information found inside the book.
</p>

<h3>🔹 Step 6: Ask Questions About the Book</h3>
<p>
The AI receives questions such as:
</p>

<ul>
<li>📖 Summarize the book.</li>
<li>🧮 Perform calculations mentioned in the book.</li>
<li>💡 Brainstorm new ideas.</li>
<li>📢 Generate marketing ideas for an AI training institute.</li>
</ul>

<h3>🔹 Step 7: AI Searches Before Answering</h3>
<p>
Whenever a question is asked, the Retrieval Agent searches the book,
finds the most relevant sections, and sends them to the AI Assistant.
</p>

<h3>🔹 Step 8: Generate Results</h3>
<p>
The AI analyzes the retrieved content and generates:
</p>

<ul>
<li>Book summaries</li>
<li>Business ideas</li>
<li>Marketing strategies</li>
<li>Python code (if required)</li>
<li>Question & Answer responses</li>
</ul>

<h3>🔹 Step 9: Monitoring with AgentOps</h3>
<p>
AgentOps tracks the conversation, agent activity, token usage, and execution flow
for debugging and monitoring purposes.
</p>

<hr>

<h3>🔄 Complete Workflow</h3>

<pre style='background:white;padding:15px;border-radius:8px;'>

PDF Book
   │
   ▼
Extract Text
   │
   ▼
Store in ChromaDB
   │
   ▼
RAG Retrieval Agent
   │
   ▼
Search Relevant Content
   │
   ▼
AI Assistant
   │
   ▼
Generate Answers / Ideas / Code
   │
   ▼
Display Results
   │
   ▼
AgentOps Monitoring

</pre>

<div style='background:#d1e7dd;padding:15px;border-radius:8px;'>
✅ In simple terms:<br><br>
The system reads a book, stores its knowledge in a searchable memory,
lets users ask questions about the book, and uses AI to generate intelligent
answers, summaries, business ideas, and code based on the book's content.
</div>

</div>
"""

display(HTML(html_content))

In [2]:
#!pip install sentence_transformers
#!pip install ag2
#!pip install fast-depends==3.0.8
#!pip install pyautogen==0.7.2
#https://microsoft.github.io/autogen/0.2/docs/notebooks/agentchat_RetrieveChat/

# Setting base for Retrive Chat

In [3]:
import json
import os

import chromadb

import autogen
from autogen import AssistantAgent
from autogen.agentchat.contrib.retrieve_user_proxy_agent import RetrieveUserProxyAgent

# Accepted file formats for that can be stored in
# a vector database instance
from autogen.retrieve_utils import TEXT_FORMATS

config_list = autogen.config_list_from_json("OAI_CONFIG_LIST.json")

assert len(config_list) > 0
print("models to use: ", [config_list[i]["model"] for i in range(len(config_list))])

Patching name='__init__', member=<function BedrockClient.__init__ at 0x0000019B5DFD6340>, patched=<function function.__call__ at 0x0000019B5DFD62A0>
Patching name='_retries', member=5, patched=5
Patching name='cost', member=<function BedrockClient.cost at 0x0000019B5DFD6660>, patched=<function function.__call__ at 0x0000019B5DFD68E0>
Patching name='create', member=<function BedrockClient.create at 0x0000019B5DFD65C0>, patched=<function function.__call__ at 0x0000019B5DFD6980>
Patching name='get_usage', member=<function BedrockClient.get_usage at 0x0000019B5DFD6700>, patched=<function function.__call__ at 0x0000019B5DFD6A20>
Patching name='message_retrieval', member=<function BedrockClient.message_retrieval at 0x0000019B5DFD63E0>, patched=<function function.__call__ at 0x0000019B5DFD6AC0>
Patching name='parse_custom_params', member=<function BedrockClient.parse_custom_params at 0x0000019B5DFD6480>, patched=<function function.__call__ at 0x0000019B5DFD6B60>
Patching name='parse_params', 

In [4]:
print("Accepted file formats for `docs_path`:")
print(TEXT_FORMATS)

Accepted file formats for `docs_path`:
['txt', 'json', 'csv', 'tsv', 'md', 'html', 'htm', 'rtf', 'rst', 'jsonl', 'log', 'xml', 'yaml', 'yml', 'pdf', 'mdx']


In [5]:
keys = {}

with open("secrets.txt", "r") as f:
    for line in f:
        key, value = line.strip().split("=", 1)
        keys[key] = value

agentops_api_key = keys["AGENTOPS_API_KEY"]
openai_api_key = keys["OPENAI_API_KEY"]

In [6]:
config_list = [
    {
        "model": "gpt-4",
        "api_key": openai_api_key,
        "api_type": "openai"
    }
]

In [7]:
import agentops
agentops.init(agentops_api_key)

🖇 AgentOps: [OPENAI INSTRUMENTOR] Error setting up OpenAI streaming wrappers: No module named 'openai.resources.beta.chat'
🖇 AgentOps: You're on the agentops free plan 🤔
--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\xsanthkum\AppData\Local\anaconda3\Lib\logging\__init__.py", line 1154, in emit
    stream.write(msg + self.terminator)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\xsanthkum\AppData\Local\anaconda3\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\U0001f914' in position 66: character maps to <undefined>
Call stack:
  File "C:\Users\xsanthkum\AppData\Local\anaconda3\Lib\runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\xsanthkum\AppData\Local\anaconda3\Lib\runpy.py", line 88, in _run_code


In [8]:
# 1. create an AssistantAgent instance named "assistant"
assistant = AssistantAgent(
    name="assistant",
    system_message="You are a helpful assistant.",
    llm_config={
        "timeout": 600,
        "cache_seed": 42,
        "config_list": config_list,
    },
)


In [9]:
from autogen.agentchat.contrib.retrieve_user_proxy_agent import RetrieveUserProxyAgent

# Extract text from local PDF files
pdf_files = ["book.pdf"]
#docs_content = [extract_text_from_pdf(file) for file in pdf_files]

from autogen.retrieve_utils import TEXT_FORMATS,extract_text_from_pdf
docs_content= extract_text_from_pdf("book.pdf")

In [10]:
# Define the RetrieveUserProxyAgent instance
ragproxyagent = RetrieveUserProxyAgent(
    name="ragproxyagent",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=3,
    retrieve_config={
        "task": "qa",
        "docs_path": ["autogen-docs/book.pdf"],
        "docs_content": docs_content,  # Provide the extracted content directly
        "chunk_token_size": 200,
        "model": config_list[0]["model"],
        "vector_db": "chroma",
        "overwrite": True,
        "get_or_create": True,
    },
    code_execution_config=False,
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# Process Starts for Scenario 1 : Generate code based off docstrings w/o human feedback

In [11]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

qa_problem = "summarize the content and use python code to any calculation given in the book"
chat_result = ragproxyagent.initiate_chat(assistant, message=ragproxyagent.message_generator, problem=qa_problem)
agentops.end_session("Success")

Trying to create collection.


max_tokens is too small to fit a single line of text. Breaking this line:
	The 22 ...
Failed to split docs with must_break_at_empty_line being True, set to False.
2026-09-17 11:32:25,811 - autogen.agentchat.contrib.retrieve_user_proxy_agent - INFO - Found 215 chunks.
2026-09-17 11:32:25,820 - autogen.agentchat.contrib.vectordb.chromadb - INFO - No content embedding is provided. Will use the VectorDB's embedding function to generate the content embedding.


VectorDB returns doc_ids:  [['fe1105cf', '8bf7f8ee', 'f0804281', '1d0894aa', '839a2e04', 'd11d708f', '150c596a', '59e93e46', '1613971f', '7091167f', '43a28a91', '16a1e2eb', 'e7d73411', '02a6f2ee', '451a2a58', 'a5e9b671', '7acf2ce9', 'a1061522', '05dac356', '466a2471']]
Adding content of doc fe1105cf to context.
Adding content of doc 8bf7f8ee to context.
Adding content of doc f0804281 to context.
Adding content of doc 1d0894aa to context.
Adding content of doc 839a2e04 to context.
Adding content of doc d11d708f to context.
Adding content of doc 150c596a to context.
Adding content of doc 59e93e46 to context.
Adding content of doc 1613971f to context.
Adding content of doc 7091167f to context.
Adding content of doc 43a28a91 to context.
Adding content of doc 16a1e2eb to context.
Adding content of doc e7d73411 to context.
Adding content of doc 02a6f2ee to context.
Adding content of doc 451a2a58 to context.
Adding content of doc a5e9b671 to context.
Adding content of doc 7acf2ce9 to context.

🖇 AgentOps: [OPENAI INSTRUMENTOR] Error setting up OpenAI streaming wrappers: No module named 'openai.resources.beta.chat'


assistant (to ragproxyagent):

The book discusses 22 laws of marketing, covering topics like the importance of being the first in the market, the power of owning a concept or word in consumers' minds, how categories divide over time, the need for adequate resources to fuel ideas, product differentiation, the influence of perception in marketing, and the importance of a good marketing strategy. Unfortunately, the text doesn't contain any specific calculations for Python code to execute.

--------------------------------------------------------------------------------


🖇 AgentOps: end_session() is deprecated and will be removed in v4 in the future. Use agentops.end_trace() instead.
🖇 AgentOps: end_session called but no active trace context found.


In [12]:
#agentops.end_session("Success")

# Process Starts for Scenario 2 : Answer a question based off docstrings w/o human feedback

In [13]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

qa_problem = "You are a creative head . do many brainstorms"
chat_result = ragproxyagent.initiate_chat(assistant, message=ragproxyagent.message_generator, problem=qa_problem)

VectorDB returns doc_ids:  [['f0804281', '7e1ac0cd', '0e08a976', '81d8712b', 'be843ac3', 'a5e9b671', 'b2f1dbea', 'ba9a27e8', '21ec236c', '51ba5a36', '8126c05a', 'cf32340a', '7e3514a1', '89fd1dac', '5b19ade7', '10e789b5', '451a2a58', 'a1f246ff', '4cf44936', 'e47994d3']]
Adding content of doc f0804281 to context.
Adding content of doc 7e1ac0cd to context.
Adding content of doc 0e08a976 to context.
Adding content of doc 81d8712b to context.
Adding content of doc be843ac3 to context.
Adding content of doc a5e9b671 to context.
Adding content of doc b2f1dbea to context.
Adding content of doc ba9a27e8 to context.
Adding content of doc 21ec236c to context.
Adding content of doc 51ba5a36 to context.
Adding content of doc 8126c05a to context.
Adding content of doc cf32340a to context.
Adding content of doc 7e3514a1 to context.
Adding content of doc 89fd1dac to context.
Adding content of doc 5b19ade7 to context.
Adding content of doc 10e789b5 to context.
Adding content of doc 451a2a58 to context.

# Process Starts for Scenario 3 : Generate code based off docstrings w/ human feedback

In [14]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

# set `human_input_mode` to be `ALWAYS`, so the agent will ask for human input at every step.
ragproxyagent.human_input_mode = "ALWAYS"
code_problem = "Based on this book give Marketing idea for AI and Datascience trainig institute. Give 5 ideas. if any coding requires use python and save it as .py"
chat_result = ragproxyagent.initiate_chat(assistant, message=ragproxyagent.message_generator, problem=code_problem)

VectorDB returns doc_ids:  [['f0804281', '81d8712b', '89fd1dac', '0e08a976', '150c596a', 'be843ac3', 'ba9a27e8', '8bf7f8ee', '8a2a101c', '039e8163', '451a2a58', 'd0e14ddf', 'b210f301', 'c1b21496', 'd5a791b1', 'a5e9b671', 'fc6e8864', '08e00728', 'ff30ed4d', '788184ee']]
Adding content of doc f0804281 to context.
Adding content of doc 81d8712b to context.
Adding content of doc 89fd1dac to context.
Adding content of doc 0e08a976 to context.
Adding content of doc 150c596a to context.
Adding content of doc be843ac3 to context.
Adding content of doc ba9a27e8 to context.
Adding content of doc 8bf7f8ee to context.
Adding content of doc 8a2a101c to context.
Adding content of doc 039e8163 to context.
Adding content of doc 451a2a58 to context.
Adding content of doc d0e14ddf to context.
Adding content of doc b210f301 to context.
Adding content of doc c1b21496 to context.
Adding content of doc d5a791b1 to context.
Adding content of doc a5e9b671 to context.
Adding content of doc fc6e8864 to context.

Replying as ragproxyagent. Provide feedback to assistant. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  Share 5 marketing strategies


ragproxyagent (to assistant):

Share 5 marketing strategies

--------------------------------------------------------------------------------


🖇 AgentOps: [agentops.InternalSpanProcessor] Error uploading logfile: Failed to upload logfile: HTTPSConnectionPool(host='api.agentops.ai', port=443): Max retries exceeded with url: /v4/logs/upload/ (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1028)')))


assistant (to ragproxyagent):

1. Leverage Success Stories: Highlight successful alumni stories to demonstrate the value of your courses and the opportunities after completing them.
2. Establish Thought Leadership: Regularly publish articles, blogs, research findings, and provide expert comments on AI and Data Science. This will establish your institute as a knowledge authority in these areas. 
3. Strategic Alliances: Partner with businesses and offer them custom training programs for their employees. This could be a new revenue stream and also a means to attract more individuals. 
4. Offer Trial Classes: Free or discounted trial classes can attract potential students to experience the teaching style and course structure before making a financial commitment.
5. Use Social Proof: Use testimonials from satisfied students, reviews, ratings on educational platforms, and word-of-mouth referrals to amplify the credibility of your institute.

--------------------------------------------------

Replying as ragproxyagent. Provide feedback to assistant. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  Any important strategy of these 5?


ragproxyagent (to assistant):

Any important strategy of these 5?

--------------------------------------------------------------------------------
assistant (to ragproxyagent):

Establishing Thought Leadership is particularly important. By regularly publishing insightful content in the field of AI and Data Science, you not only provide value to the community but also position your institute as an authority in these areas. This can attract students who seek high-quality and credible education.

--------------------------------------------------------------------------------


Replying as ragproxyagent. Provide feedback to assistant. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  exit


# Process Starts for Scenario 4: Answer a question based off docstrings w/ human feedback

In [15]:
# reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

# set `human_input_mode` to be `ALWAYS`, so the agent will ask for human input at every step.
ragproxyagent.human_input_mode = "ALWAYS"
qa_problem = "Based on this book give Marketing idea for AI and Datascience trainig institute. Give 5 ideas. "
chat_result = ragproxyagent.initiate_chat(
    assistant, message=ragproxyagent.message_generator, problem=qa_problem
)  # type "exit" to exit the conversation

VectorDB returns doc_ids:  [['f0804281', '81d8712b', '89fd1dac', 'be843ac3', '150c596a', 'b210f301', '8a2a101c', 'ba9a27e8', '0e08a976', '788184ee', '039e8163', '451a2a58', '4e3d050e', 'a1f246ff', 'd5a791b1', '08e00728', '083871f5', 'b99eaa4e', 'c1b21496', 'ff30ed4d']]
Adding content of doc f0804281 to context.
Adding content of doc 81d8712b to context.
Adding content of doc 89fd1dac to context.
Adding content of doc be843ac3 to context.
Adding content of doc 150c596a to context.
Adding content of doc b210f301 to context.
Adding content of doc 8a2a101c to context.
Adding content of doc ba9a27e8 to context.
Adding content of doc 0e08a976 to context.
Adding content of doc 788184ee to context.
Adding content of doc 039e8163 to context.
Adding content of doc 451a2a58 to context.
Adding content of doc 4e3d050e to context.
Adding content of doc a1f246ff to context.
Adding content of doc d5a791b1 to context.
Adding content of doc 08e00728 to context.
Adding content of doc 083871f5 to context.

Replying as ragproxyagent. Provide feedback to assistant. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  Share some important pointers from the book


ragproxyagent (to assistant):

Share some important pointers from the book

--------------------------------------------------------------------------------
assistant (to ragproxyagent):

1. Marketing follows inherent laws: Understanding and following these 22 laws can help a product or service become successful in the marketplace.
2. Importance of First: Being the first in the market, or the first in a consumer's mind, is often more advantageous than being the best.
3. Perception matters: Marketing is not a battle of products, but a battle of perceptions.
4. Value of resources: Even the best ideas require funding and resources to lift off.
5. Admitting position: Acknowledging your current position in the market can often lead to progress.
6. Don't predict the future: Marketing plans should not be based on predictions of the future, because the future is inherently unpredictable.
7. Don't get arrogant: Success can often lead to arrogance, which is an enemy of effective marketing. Succe

Replying as ragproxyagent. Provide feedback to assistant. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  exit
